In [ ]:
#importing tools and technologies
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

#Loading the DataSet
df=pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

#Checking Dataset Shape and Structure
print("--- Dataset Overview ---")
print(f"Dataset Shape: {df.shape}")
print("\n--- Data Info ---")
print(df.info())
print("\n--- First 5 Rows ---")
print(df.head())




In [ ]:
#handling missing values
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())
df = df.drop('customerID', axis=1)


In [ ]:
#Analyzing the numerical and categorical features
numerical_features = df.select_dtypes(include=['int64', 'float64']).columns
categorical_features = df.select_dtypes(include=['object']).columns
print("\n--- Numerical Features Summary ---")
print("Numerical Features:", numerical_features)
print("\n--- Categorical Features Summary ---")
print("Categorical Features:", categorical_features)
df[numerical_features].describe()
df[categorical_features].describe() 

#Visualizing the churn distribution
df['Churn'].value_counts()
plt.figure(figsize=(6,4))
sns.countplot(x='Churn', data=df)
plt.title('Customer Churn Distribution')
plt.xlabel('Churn (0=No, 1=Yes)')
plt.ylabel('Number of Customers')
plt.show()

#converting categorical variables using label endoding/ one-hot encoding
label_encoder= LabelEncoder()
for column in categorical_features:
    df[column] = label_encoder.fit_transform(df[column])

#feature scaling and StandardScaler
x=df.drop('Churn', axis=1)
y=df['Churn']
scaler=StandardScaler()
x_scaled=scaler.fit_transform(x)

#Splitting the data into training and testing sets
x_train, x_test, y_train, y_test = train_test_split(x_scaled, y, test_size=0.2, random_state=42)

                    #Model Building
#Logistic Regression
lr= LogisticRegression(max_iter=1000)
lr.fit(x_train, y_train)
lr_pred= lr.predict(x_test)

#Decision Tree Classifier
dt= DecisionTreeClassifier()
dt.fit(x_train, y_train)
dt_pred= dt.predict(x_test)

#Random Forest Classifier
rf= RandomForestClassifier()
rf.fit(x_train, y_train)
rf_pred= rf.predict(x_test)

#Support Vector Machine
svm= SVC(kernel='rbf', random_state=42)
svm.fit(x_train, y_train)
svm_pred= svm.predict(x_test)

#model evaluation
def evaluate_model(name, y_test, y_pred):
    acc = accuracy_score(y_test, y_pred)
    prec= precision_score(y_test, y_pred)
    rec= recall_score(y_test, y_pred)
    f1= f1_score(y_test, y_pred)

    print(f"\n{name} performance:")
    print("Accuracy:", acc)
    print("Recall:", rec)
    print("Precision:",prec)
    print("F1-Score: ",f1)

    return acc, prec, rec, f1

results={}
results['Logistic Regression']= evaluate_model(
    "Logistic Regression", y_test, lr_pred
)
results['Decision Tree']=evaluate_model(
    "Decision Tree", y_test, dt_pred
)
results['Random Forest']=evaluate_model(
    "Random Forest", y_test, rf_pred
)
results['Support Vector Machine'] = evaluate_model(
    "Support Vector Machine", y_test, svm_pred
)

print("Logistic Regression Accuracy:", accuracy_score(y_test, lr_pred))
print("Decision Tree Accuracy:", accuracy_score(y_test, dt_pred))
print("Random Forest Accuracy:", accuracy_score(y_test, rf_pred))
print("SVM Accuracy:", accuracy_score(y_test, svm_pred))

print("Random Forest Classification Report:\n", classification_report(y_test, rf_pred))

#confusion matrix
models={
    "LogisticRegression": lr_pred,
    "Decision Tree":dt_pred,
    "Random Forest": rf_pred,
}
for name, pred in models.items():
    cm=confusion_matrix(y_test, pred)

    plt.figure(figsize=(4,3))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f"Confusion Matrix - {name}")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()

#Model Comaprison
comparison_df = pd.DataFrame.from_dict(results, orient='index', columns=['Accuracy', 'Precision', 'Recall', 'F1-Score'])
print("\n--- Model Comparison ---")
print(comparison_df)

best_model = comparison_df['F1-Score'].idxmax()
print(f"\nBest Model based on F1-Score: {best_model}")

#final classification report for the best model
print(f"\nClassification Report for {best_model}:\n")
if best_model == "Logistic Regression":
    print(classification_report(y_test, lr_pred))
elif best_model == "Decision Tree":
    print(classification_report(y_test, dt_pred))
elif best_model == "Random Forest":
    print(classification_report(y_test, rf_pred))

#sample prediction using the best model
sample_customer=np.array([[1 , 0, 1, 45, 1, 0, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 0, 75, 3500]])
sample_customer_df = pd.DataFrame(sample_customer, columns=x.columns)
sample_customer_scaled= scaler.transform(sample_customer_df)

prediction = rf.predict(sample_customer_scaled)

if prediction[0]==1:
    print("\nPrediction: Customer is likely to CHURN")
else:
    print("\nPrediction Customer is NOT likely to churn")
